# Refined Benchmark

Two modes controlled by `LOAD_BASELINE`:
- `LOAD_BASELINE = True` — loads generated code + metrics from an existing baseline output dir, then runs analysis → refinement → evaluation. No initial LLM call.
- `LOAD_BASELINE = False` — runs the full pipeline: initial LLM call → evaluate → analyze → refine → evaluate.

In [1]:
import importlib
import llm as llm_module
import filter as filter_module
importlib.reload(llm_module)
importlib.reload(filter_module)
from llm import call_llm, build_prompt
from filter import filter_services

import evaluate as evaluate_module
import analysis as analysis_module
from evaluate import evaluate, print_metrics, print_evaluation_summary
from benchmarks import load_benchmarks
from analysis import analyze
import output as output_module
from output import load_baseline_results

## Configuration

In [2]:
from pathlib import Path
BENCHMARK_DIR = Path("./benchmark")
BENCHMARK_TYPES = ["socbenchd_1"]  # add more benchmark types here
BENCHMARK_LIMIT = None  # set to None for all sectors
QUERY_LIMIT = 50        # set to None for all queries (ignored when LOAD_BASELINE=True)
MAX_WORKERS = 10
MODEL = "deepseek-ai/DeepSeek-V4-Pro"

FILTER_SERVICES = True  # BM25 endpoint pre-filtering
TOP_K = 5               # endpoints to keep per service when FILTER_SERVICES=True

LOAD_BASELINE = True   # True = load saved baseline from disk; False = run initial LLM call fresh
BASELINE_DIR = "output/2026-06-28_17-32-41_baseline"  # only used when LOAD_BASELINE=True

benchmark_sets, total_available, total_queries_available = load_benchmarks(BENCHMARK_DIR, BENCHMARK_TYPES, BENCHMARK_LIMIT, None)
total_queries = min(QUERY_LIMIT, total_queries_available) if QUERY_LIMIT is not None else total_queries_available
print(f"Loaded {len(benchmark_sets)} benchmark sets (total available: {total_available})")
if not LOAD_BASELINE:
    print(f"Total queries: {total_queries} (of {total_queries_available} available)")
print(f"Mode: {'LOAD BASELINE from ' + BASELINE_DIR if LOAD_BASELINE else 'FULL (run initial + refine)'} | Workers: {MAX_WORKERS} | Model: {MODEL}")
print(f"Filter: {FILTER_SERVICES} (top_k={TOP_K})")

Loaded 11 benchmark sets (total available: 11)
Mode: LOAD BASELINE from output/2026-06-28_17-32-41_baseline | Workers: 10 | Model: deepseek-ai/DeepSeek-V4-Pro
Filter: True (top_k=5)


## Prompt Template

Used when `LOAD_BASELINE = False`. When loading a baseline the prompt is read from `prompt.txt` in the baseline output directory.

In [3]:
PROMPT_TEMPLATE = '''You are an expert software engineer performing REST service composition.

You are given:


A natural-language task description.
One or more REST API specifications (OpenAPI/Swagger).


Your job is to write a single self-contained Python script that fulfills the task by calling the necessary endpoints, in the correct order, passing data from earlier responses into later requests as required.

Output contract


Respond with raw Python source code ONLY. Your entire response must be directly executable by a Python interpreter with no edits.
The response must begin with an import statement (e.g. import requests). Do not emit markdown, code fences (```), backticks, language tags, comments, docstrings, prose, or trailing notes — nothing but code.
Import requests and define exactly one function named compose. Place all request logic inside compose. Do NOT call compose.
compose must return the final result that answers the task.


Composition rules


Use ONLY endpoints, HTTP methods, paths, parameters, and fields defined in the provided specifications. Do not invent endpoints, parameters, response fields, or hosts.
Build each request URL by joining the base URL from the spec\'s servers field with the operation path. If several servers are listed, use the first.
Place parameters exactly as the spec defines them: substitute path parameters into the URL, pass query parameters via params=, headers via headers=, and request bodies via json= (or data= for form bodies). Use the exact parameter and field names from the spec.
If the spec defines a security scheme (API key, bearer token, etc.), include it where the spec requires it (header or query). When no concrete value is supplied in the task, use a clearly named placeholder constant (e.g. API_KEY = "<API_KEY>").
Parse JSON responses with .json(), extract the specific fields you need, and feed them into subsequent calls. Chain calls so each step\'s output drives the next.
Iterate when the task requires processing a collection; otherwise issue each required call once.
Call only the endpoints strictly required to satisfy the task. Do not call supplementary endpoints whose output is not used as input to a later step or as the final result. When two endpoints seem relevant to the same task step, pick the one whose description most directly matches — do not call both.


Code-quality rules (the output is statically analyzed)


Use only the requests library and the Python standard library. No other third-party imports.
Every name must be defined before use. No undefined references, no unused imports or variables, no placeholders like ... or TODO.
Add type annotations to the compose signature and its return type, and to local variables where the type is clear. Import every typing symbol you reference (from typing import Any). Annotate decoded JSON as Any or dict[str, Any] rather than guessing concrete shapes.
Write valid, parseable Python with explicit, deterministic control flow.


Output shape (format example only — do NOT copy its logic or endpoints)

import requests
from typing import Any

def compose() -> Any:
    base_url = "https://api.example.com"
    first = requests.get(f"{{base_url}}/items", params={{"limit": 1}}).json()
    item_id = first["data"][0]["id"]
    detail = requests.get(f"{{base_url}}/items/{{item_id}}").json()
    return detail

Task

{query}

Source

{services_block}
'''


## Initial LLM Call  (or Load Baseline)

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

if LOAD_BASELINE:
    importlib.reload(output_module)
    sector_results = load_baseline_results(BASELINE_DIR)
    sector_results.sort(key=lambda r: (r['sector_name'], r['query_index']))
    print(f"Loaded {len(sector_results)} results from {BASELINE_DIR}")
else:
    tasks = []
    for benchmark in benchmark_sets:
        if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
            break
        for query_index, query in enumerate(benchmark['queries'], start=1):
            if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
                break
            tasks.append((benchmark, query_index, query))

    def _call_initial(args):
        benchmark, query_index, query = args
        services = (
            filter_services(benchmark['services'], query['query'], top_k=TOP_K)
            if FILTER_SERVICES else benchmark['services']
        )
        prompt = build_prompt(services, query['query'], PROMPT_TEMPLATE)
        t0 = time.time()
        generated, usage = call_llm(prompt, MODEL, '')
        elapsed = time.time() - t0
        generated += '\n\ncompose()'
        return {
            'query_index': query_index,
            'sector_name': benchmark['name'],
            'query': query,
            'prompt': prompt,
            'generated': generated,
            'service_files': benchmark.get('service_files', []),
            'model': MODEL,
            '_elapsed': elapsed,
            '_usage': usage,
        }

    sector_results = []
    call_times = []
    run_start = time.time()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_call_initial, t): t for t in tasks}
        for future in as_completed(futures):
            result = future.result()
            sector_results.append(result)
            call_times.append(result['_elapsed'])

            done = len(sector_results)
            avg_t = sum(call_times) / len(call_times)
            remaining = total_queries - done
            eta_s = remaining * avg_t / min(remaining, MAX_WORKERS) if remaining else 0
            usage = result['_usage']
            tok_str = (f"prompt={usage['prompt_tokens']} comp={usage['completion_tokens']}"
                       if usage else "tokens=n/a")
            print(
                f"[{done:>3}/{total_queries}] [{result['sector_name']}] Q{result['query_index']}"
                f" | {result['_elapsed']:.1f}s | {tok_str}"
                f" | avg {avg_t:.1f}s | ETA ~{eta_s:.0f}s"
            )

    sector_results.sort(key=lambda r: (r['sector_name'], r['query_index']))
    print(f"\nDone. Total wall time: {time.time() - run_start:.1f}s")

Loaded 50 results from output/2026-06-28_17-32-41_baseline


## Evaluate Initial

Skipped when loading a baseline — metrics are already stored in the loaded results.

In [5]:
if not LOAD_BASELINE:
    for result in sector_results:
        initial_metrics = evaluate(result['generated'], result['query'].get('endpoints', []))
        result['initial_metrics'] = initial_metrics
        print_metrics(initial_metrics, f"Initial evaluation - Query {result['query_index']}")
else:
    print("[LOAD_BASELINE] Using pre-computed initial metrics from disk.")

[LOAD_BASELINE] Using pre-computed initial metrics from disk.


## Analyze

In [6]:
importlib.reload(analysis_module)
for result in sector_results:
    analysis = analyze(result['generated'], service_files=result.get('service_files'), benchmark_dir=BENCHMARK_DIR)
    result['analysis'] = analysis
    print(f"Analysis - [{result['sector_name']}] Query {result['query_index']}\n", analysis)

Analysis - [01-energy] Query 1
 AST: No syntax errors found.
Ruff: F841 Local variable `equipment_base_url` is assigned to but never used
Ruff:  --> /var/folders/57/p3z5bqsd11l937yr1409c3m00000gn/T/tmpmtp4yimd/generated.py:6:5
Ruff:   |
Ruff: 4 | def compose() -> Any:
Ruff: 5 |     # Step 1: Retrieve equipment status and performance metrics
Ruff: 6 |     equipment_base_url = "https://api.example.com/v1"  # Placeholder from Carbon Footprint spec, but equipment status exists in Energy …
Ruff:   |     ^^^^^^^^^^^^^^^^^^
Ruff: 7 |     # Actually, the first spec has /equipment-status without a server, so use placeholder
Ruff: 8 |     eq_resp = requests.get("https://api.example.com/equipment-status")
Ruff:   |
Ruff: help: Remove assignment to unused variable `equipment_base_url`
Ruff: Found 1 error.
Ruff: No fixes available (1 hidden fix can be enabled with the `--unsafe-fixes` option).
Httpretty: GET https://api.example.com/equipment-status
Httpretty: GET https://api.example.com/alerts
Http

## Refinement LLM Call

In [7]:
def _call_refined(result):
    original_code = result['generated'].removesuffix('\n\ncompose()')
    query_text = result['query']['query']
    services = [(Path(BENCHMARK_DIR) / sf).read_text() for sf in result.get('service_files', [])]
    if FILTER_SERVICES:
        services = filter_services(services, query_text, top_k=TOP_K)
    services_block = "\n---\n".join(services)
    refined_prompt = f'''
You are improving Python REST service-composition code that you previously generated. A review of that code from static-analysis tools is provided. Use it to repair the code.

Inputs

Analysis: findings from the static-analysis and review tools (e.g. Ruff, mypy, and spec-conformance checks). Treat each finding as an issue to resolve.
Original code: the program to repair.
Task and Source: the task the code must fulfill and the API specification(s) it may use. Use these to fix correctness and spec-conformance findings.

What to do

Resolve every issue raised in the analysis, plus any clear correctness bug you notice.
Preserve the existing correct behavior. Make the minimum changes needed — do not rewrite working logic or restructure code the analysis did not flag.
Fix issues genuinely. Do not suppress diagnostics (no # type: ignore, no broad except, no deleting required functionality) merely to silence the review.
Keep using only the endpoints, methods, parameters, and fields defined in the specification; do not invent any.
Call only the endpoints strictly required to satisfy the task. If the original code calls supplementary endpoints whose output is not used as input to a later step or as the final result, remove those calls. When two endpoints seem relevant to the same task step, keep only the one whose description most directly matches — do not call both.
If the analysis reports no actionable issues, return the original code unchanged.

Output contract (identical to the generation step)

Respond with raw Python source code ONLY: the complete, corrected program — not a diff, patch, or snippet. Your entire response must run as-is.
The response must begin with an import statement. No markdown, code fences (```), backticks, language tags, comments, docstrings, prose, or notes — nothing but code.
Import requests and define exactly one function named compose containing all request logic. Do NOT call compose. compose must return the final result that answers the task.
Use only the requests library and the Python standard library. Every name defined before use; no unused imports or variables; no ... or TODO.
Keep type annotations on the compose signature and return type; import every typing symbol you reference.

Output shape (format example only — do NOT copy its logic)

import requests
from typing import Any

def compose() -> Any:
    base_url = "https://api.example.com"
    first = requests.get(f"{{base_url}}/items", params={{"limit": 1}}).json()
    item_id = first["data"][0]["id"]
    detail = requests.get(f"{{base_url}}/items/{{item_id}}").json()
    return detail

Analysis

{result['analysis']}

Original code

{original_code}

Task

{query_text}

Source

{services_block}
'''
    generated_refined, _usage = call_llm(refined_prompt, MODEL, '')
    generated_refined += '\n\ncompose()'
    result['generated_refined'] = generated_refined
    return result

refined_count = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_call_refined, r): r for r in sector_results}
    for future in as_completed(futures):
        future.result()
        refined_count += 1
        print(f"[{refined_count}/{len(sector_results)}] Refinement done")

[1/50] Refinement done
[2/50] Refinement done
[3/50] Refinement done
[4/50] Refinement done
[5/50] Refinement done
[6/50] Refinement done
[7/50] Refinement done
[8/50] Refinement done
[9/50] Refinement done
[10/50] Refinement done
[11/50] Refinement done
[12/50] Refinement done
[13/50] Refinement done
[14/50] Refinement done
[15/50] Refinement done
[16/50] Refinement done
[17/50] Refinement done
[18/50] Refinement done
[19/50] Refinement done
[20/50] Refinement done
[21/50] Refinement done
[22/50] Refinement done
[23/50] Refinement done
[24/50] Refinement done
[25/50] Refinement done
[26/50] Refinement done
[27/50] Refinement done
[28/50] Refinement done
[29/50] Refinement done
[30/50] Refinement done
[31/50] Refinement done
[32/50] Refinement done
[33/50] Refinement done
[34/50] Refinement done
[35/50] Refinement done
[36/50] Refinement done
[37/50] Refinement done
[38/50] Refinement done
[39/50] Refinement done
[40/50] Refinement done
[41/50] Refinement done
[42/50] Refinement done
[

## Evaluate Refined

In [8]:
for result in sector_results:
    refined_metrics = evaluate(result['generated_refined'], result['query'].get('endpoints', []))
    result['refined_metrics'] = refined_metrics
    print_metrics(refined_metrics, f"Refined evaluation - [{result['sector_name']}] Query {result['query_index']}")

Refined evaluation - [01-energy] Query 1
  Precision: 1.00
  Recall:    1.00
  F1:        1.00
  Extracted: ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Expected:  ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Missing:   []
  Extra:     []
Refined evaluation - [01-energy] Query 2
  Precision: 1.00
  Recall:    1.00
  F1:        1.00
  Extracted: ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Expected:  ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Missing:   []
  Extra:     []
Refined evaluation - [01-energy] Query 3
  Precision: 0.67
  Recall:    0.75
  F1:        0.71
  Extracted: ['GET /energy-patterns', 'GET /equipment-monitoring', 'G

## Save Outputs

In [9]:
import os
from datetime import datetime
importlib.reload(output_module)

if LOAD_BASELINE:
    baseline_tag = os.path.basename(BASELINE_DIR)
    mode_tag = f"refined_from_{baseline_tag}"
else:
    mode_tag = "refined"

run_name = datetime.now().strftime('%Y-%m-%d_%H-%M-%S') + f"_{mode_tag}"
outdir = output_module.make_output_dir(run_name)
for r in sector_results:
    output_module.write_query_output(outdir, r)

run_config = {
    'benchmark_dir': str(BENCHMARK_DIR),
    'benchmark_types': BENCHMARK_TYPES,
    'benchmark_limit': BENCHMARK_LIMIT,
    'query_limit': QUERY_LIMIT,
    'baseline': False,
    'model': MODEL,
    'load_baseline': LOAD_BASELINE,
    'baseline_dir': BASELINE_DIR if LOAD_BASELINE else None,
    'filter_services': FILTER_SERVICES,
    'top_k': TOP_K if FILTER_SERVICES else None,
}
output_module.write_overall_summary(outdir, sector_results, run_config)
print('Wrote outputs to', outdir)

Wrote outputs to output/2026-06-28_17-36-20_refined_from_2026-06-28_17-32-41_baseline


## Summary

In [10]:
print_evaluation_summary([result['initial_metrics'] for result in sector_results], "Initial Evaluation Summary")
print_evaluation_summary([result['refined_metrics'] for result in sector_results], "Refined Evaluation Summary")

Initial Evaluation Summary
  Average Precision: 0.74
  Average Recall:    0.82
  Average F1:        0.77
  Avg. Missing Endpoints: 1.72
  Avg. Extra Endpoints:   2.34
  Correct Compositions: 4/50 (8.0%)
Refined Evaluation Summary
  Average Precision: 0.75
  Average Recall:    0.83
  Average F1:        0.78
  Avg. Missing Endpoints: 1.70
  Avg. Extra Endpoints:   2.30
  Correct Compositions: 4/50 (8.0%)
